# ❤️ Heart Disease Prediction — GeoAI Edition
### Task 3 — Classification + Geospatial Risk Mapping

| Item | Details |
|------|---------|
| **Dataset** | Heart Disease UCI (adapted with Pakistan city data) |
| **Models** | Logistic Regression + Decision Tree |
| **Geospatial** | Risk map across 10 major Pakistan cities |
| **Evaluation** | Accuracy, ROC Curve, Confusion Matrix, Feature Importance |

---
### What Makes This GeoAI?
- Dataset includes **real Pakistan city coordinates**
- **Urban stress index** per city as a geospatial feature
- **Interactive Folium map** showing heart disease risk by city
- **Choropleth-style** risk visualization across Pakistan

---
## Step 1: Install Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn folium

---
## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    roc_curve, auc, classification_report
)
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported!')

---
## Step 3: Load Dataset
Dataset contains UCI heart disease features + Pakistan city geospatial data.

In [ ]:
df = pd.read_csv('heart_disease_pakistan.csv')

print('✅ Dataset loaded!')
print(f'   Shape: {df.shape}')
print(f'   Columns: {list(df.columns)}')
print()
display(df.head())

---
## Step 4: Data Cleaning & Inspection

In [ ]:
print('📋 Dataset Info:')
print(df.info())
print()
print('📊 Summary Statistics:')
display(df.describe())
print()
print('Missing values:')
print(df.isnull().sum())
print('\n✅ No missing values — dataset is clean!')

---
## Step 5: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Heart Disease EDA — Feature Distributions', fontsize=15, fontweight='bold')

# Age distribution
sns.histplot(data=df, x='age', hue='target', bins=20, ax=axes[0,0], palette=['#2ecc71','#e74c3c'])
axes[0,0].set_title('Age Distribution by Heart Disease')
axes[0,0].legend(['No Disease','Heart Disease'])

# Cholesterol
sns.boxplot(data=df, x='target', y='chol', ax=axes[0,1], palette=['#2ecc71','#e74c3c'])
axes[0,1].set_title('Cholesterol by Heart Disease')
axes[0,1].set_xticklabels(['No Disease','Heart Disease'])

# Blood pressure
sns.boxplot(data=df, x='target', y='trestbps', ax=axes[0,2], palette=['#2ecc71','#e74c3c'])
axes[0,2].set_title('Blood Pressure by Heart Disease')
axes[0,2].set_xticklabels(['No Disease','Heart Disease'])

# Max heart rate
sns.histplot(data=df, x='thalach', hue='target', bins=20, ax=axes[1,0], palette=['#2ecc71','#e74c3c'])
axes[1,0].set_title('Max Heart Rate Distribution')

# Target distribution
counts = df['target'].value_counts()
axes[1,1].pie(counts, labels=['Heart Disease','No Disease'],
              colors=['#e74c3c','#2ecc71'], autopct='%1.1f%%',
              startangle=90, textprops={'fontsize':11})
axes[1,1].set_title('Target Distribution')

# Urban stress vs target
sns.boxplot(data=df, x='target', y='urban_stress_index', ax=axes[1,2], palette=['#2ecc71','#e74c3c'])
axes[1,2].set_title('Urban Stress Index by Heart Disease')
axes[1,2].set_xticklabels(['No Disease','Heart Disease'])

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved as eda_plots.png')

In [ ]:
# Heart disease rate by Pakistan city
city_risk = df.groupby('city')['target'].agg(['mean','count']).reset_index()
city_risk.columns = ['city','risk_rate','count']
city_risk = city_risk.sort_values('risk_rate', ascending=False)

plt.figure(figsize=(12, 5))
bars = plt.bar(city_risk['city'], city_risk['risk_rate']*100,
               color=plt.cm.RdYlGn_r(city_risk['risk_rate']), edgecolor='black')
for bar, rate in zip(bars, city_risk['risk_rate']):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{rate*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
plt.title('Heart Disease Risk Rate by Pakistan City', fontsize=14, fontweight='bold')
plt.xlabel('City'); plt.ylabel('Risk Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('city_risk_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ City risk chart saved')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
numeric_cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
                'thalach','exang','oldpeak','slope','ca','thal',
                'urban_stress_index','target']
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 6: Data Preprocessing

In [ ]:
# Features and target
feature_cols = ['age','sex','cp','trestbps','chol','fbs','restecg',
                'thalach','exang','oldpeak','slope','ca','thal','urban_stress_index']

X = df[feature_cols]
y = df['target']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Data preprocessed!')
print(f'   Training samples: {len(X_train)}')
print(f'   Testing samples:  {len(X_test)}')
print(f'   Features used:    {len(feature_cols)} (including urban_stress_index)')

---
## Step 7: Model 1 — Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred  = lr_model.predict(X_test_scaled)
lr_prob  = lr_model.predict_proba(X_test_scaled)[:,1]
lr_acc   = accuracy_score(y_test, lr_pred)

print('📊 Logistic Regression Results:')
print(f'   Accuracy: {lr_acc:.4f} ({lr_acc*100:.2f}%)')
print()
print(classification_report(y_test, lr_pred, target_names=['No Disease','Heart Disease']))

---
## Step 8: Model 2 — Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
dt_pred  = dt_model.predict(X_test)
dt_prob  = dt_model.predict_proba(X_test)[:,1]
dt_acc   = accuracy_score(y_test, dt_pred)

print('📊 Decision Tree Results:')
print(f'   Accuracy: {dt_acc:.4f} ({dt_acc*100:.2f}%)')
print()
print(classification_report(y_test, dt_pred, target_names=['No Disease','Heart Disease']))

---
## Step 9: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

for ax, pred, title in zip(axes,
    [lr_pred, dt_pred],
    ['Logistic Regression', 'Decision Tree']):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
                xticklabels=['No Disease','Heart Disease'],
                yticklabels=['No Disease','Heart Disease'], ax=ax,
                linewidths=1, linecolor='white')
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test,pred)*100:.2f}%',
                 fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved')

---
## Step 10: ROC Curves

In [ ]:
plt.figure(figsize=(9, 6))

for prob, label, color in [
    (lr_prob, 'Logistic Regression', '#3498db'),
    (dt_prob, 'Decision Tree',       '#e74c3c')
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2.5,
             label=f'{label} (AUC = {roc_auc:.3f})')

plt.plot([0,1],[0,1],'k--',linewidth=1.5,label='Random Classifier')
plt.fill_between([0,1],[0,1], alpha=0.05, color='gray')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Model Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curves saved')

---
## Step 11: Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold')

# Logistic Regression coefficients
lr_coef = pd.Series(np.abs(lr_model.coef_[0]), index=feature_cols).sort_values()
colors_lr = ['#e74c3c' if f == 'urban_stress_index' else '#3498db' for f in lr_coef.index]
lr_coef.plot(kind='barh', ax=axes[0], color=colors_lr, edgecolor='black')
axes[0].set_title('Logistic Regression\nFeature Coefficients (|value|)')
axes[0].set_xlabel('Absolute Coefficient')

# Decision Tree importance
dt_imp = pd.Series(dt_model.feature_importances_, index=feature_cols).sort_values()
colors_dt = ['#e74c3c' if f == 'urban_stress_index' else '#2ecc71' for f in dt_imp.index]
dt_imp.plot(kind='barh', ax=axes[1], color=colors_dt, edgecolor='black')
axes[1].set_title('Decision Tree\nFeature Importance Score')
axes[1].set_xlabel('Importance Score')

legend = [mpatches.Patch(color='#e74c3c', label='Geospatial Feature'),
          mpatches.Patch(color='#3498db', label='Clinical Feature (LR)'),
          mpatches.Patch(color='#2ecc71', label='Clinical Feature (DT)')]
fig.legend(handles=legend, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5,-0.05), fontsize=10)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance saved')

---
## Step 12: Model Comparison

In [ ]:
from sklearn.metrics import roc_auc_score

lr_fpr,lr_tpr,_ = roc_curve(y_test,lr_prob)
dt_fpr,dt_tpr,_ = roc_curve(y_test,dt_prob)

comparison = pd.DataFrame({
    'Model': ['Logistic Regression','Decision Tree'],
    'Accuracy (%)': [round(lr_acc*100,2), round(dt_acc*100,2)],
    'ROC-AUC': [round(auc(lr_fpr,lr_tpr),4), round(auc(dt_fpr,dt_tpr),4)]
})

print('📊 MODEL COMPARISON')
print('='*45)
print(comparison.to_string(index=False))
print('='*45)
winner = 'Logistic Regression' if lr_acc >= dt_acc else 'Decision Tree'
print(f'\n🏆 Best Model: {winner}')

---
## Step 13: GeoAI — Interactive Risk Map
Heart disease risk mapped across 10 major Pakistan cities using Folium.

In [ ]:
# Calculate risk stats per city
city_stats = df.groupby(['city','latitude','longitude','province']).agg(
    risk_rate=('target','mean'),
    total_patients=('target','count'),
    avg_age=('age','mean'),
    avg_chol=('chol','mean'),
    avg_bp=('trestbps','mean'),
    urban_stress=('urban_stress_index','mean')
).reset_index()
city_stats['risk_pct'] = (city_stats['risk_rate']*100).round(1)
city_stats['avg_age']  = city_stats['avg_age'].round(1)
city_stats['avg_chol'] = city_stats['avg_chol'].round(1)
city_stats['avg_bp']   = city_stats['avg_bp'].round(1)

print('✅ City risk statistics:')
display(city_stats[['city','province','risk_pct','total_patients','avg_age','avg_chol','urban_stress']])

In [ ]:
def build_risk_map(city_stats):
    m = folium.Map(location=[30.0, 70.0], zoom_start=6, tiles=None)

    # Satellite layer
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Satellite', overlay=False, control=True
    ).add_to(m)
    folium.TileLayer(tiles='CartoDB positron', name='Light Map',
        overlay=False, control=True).add_to(m)

    # Risk markers
    risk_group = folium.FeatureGroup(name='Heart Disease Risk by City')
    for _, row in city_stats.iterrows():
        risk = row['risk_pct']
        # Color based on risk level
        if risk >= 90:   color = '#8B0000'
        elif risk >= 80: color = '#cc0000'
        elif risk >= 70: color = '#ff4444'
        elif risk >= 60: color = '#ff8800'
        else:            color = '#ffcc00'
        radius = 15 + (row['total_patients'] / 10)

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=radius, color=color,
            fill=True, fill_color=color, fill_opacity=0.75,
            tooltip=f"{row['city']} — Risk: {risk}%",
            popup=folium.Popup(
                f"<div style='font-family:Arial;width:220px;'>"
                f"<b style='font-size:14px;'>{row['city']}</b>"
                f"<span style='color:#888;'> ({row['province']})</span><br><br>"
                f"<b style='color:#cc0000;'>Heart Disease Risk: {risk}%</b><br>"
                f"Total Patients: {row['total_patients']}<br>"
                f"Avg Age: {row['avg_age']} yrs<br>"
                f"Avg Cholesterol: {row['avg_chol']} mg/dl<br>"
                f"Avg Blood Pressure: {row['avg_bp']} mmHg<br>"
                f"Urban Stress Index: {row['urban_stress']}<br><br>"
                f"<span style='font-size:10px;color:#888;'>Click for details</span></div>",
                max_width=240)
        ).add_to(risk_group)

        # City label
        folium.Marker(
            location=[row['latitude']+0.3, row['longitude']],
            icon=folium.DivIcon(html=
                f"<div style='font-family:Arial;font-size:11px;"
                f"font-weight:bold;color:white;text-shadow:1px 1px 2px black;"
                f"white-space:nowrap;'>{row['city']}<br>{risk}%</div>")
        ).add_to(risk_group)
    risk_group.add_to(m)

    # Title
    m.get_root().html.add_child(folium.Element("""
    <div style='position:fixed;top:12px;left:50%;transform:translateX(-50%);
        background:rgba(255,255,255,0.95);color:#cc0000;
        padding:10px 22px;border-radius:10px;
        border:2px solid #cc0000;font-family:Arial;
        font-size:15px;font-weight:bold;z-index:9999;text-align:center;
        box-shadow:0 4px 12px rgba(0,0,0,0.2);'>
        ❤️ Heart Disease Risk Map — Pakistan
        <div style='font-size:10px;color:#666;margin-top:3px;'>
        Based on Clinical + Geospatial Analysis | Bubble size = patient count</div>
    </div>"""))

    # Risk legend
    m.get_root().html.add_child(folium.Element("""
    <div style='position:fixed;bottom:35px;left:12px;
        background:rgba(255,255,255,0.95);color:#333;
        padding:14px 18px;border-radius:10px;
        border:1px solid #ccc;font-family:Arial;
        font-size:12px;z-index:9999;
        box-shadow:0 4px 12px rgba(0,0,0,0.15);'>
        <b style='color:#cc0000;'>Risk Level</b><br><br>
        <span style='color:#8B0000;font-size:18px;'>●</span> Very High (≥90%)<br>
        <span style='color:#cc0000;font-size:18px;'>●</span> High (80–90%)<br>
        <span style='color:#ff4444;font-size:18px;'>●</span> Medium-High (70–80%)<br>
        <span style='color:#ff8800;font-size:18px;'>●</span> Medium (60–70%)<br>
        <span style='color:#ffcc00;font-size:18px;'>●</span> Lower (&lt;60%)<br><br>
        <span style='color:#888;font-size:10px;'>Bubble size = patient count<br>Click for full details</span>
    </div>"""))

    folium.LayerControl().add_to(m)
    m.save('heart_disease_risk_map.html')
    return m

print('✅ Map function ready!')

In [ ]:
risk_map = build_risk_map(city_stats)
print('✅ Map saved: heart_disease_risk_map.html')
print('   Click any bubble for full city health statistics!')
risk_map

---
## Step 14: Decision Tree Visualization

In [ ]:
plt.figure(figsize=(20, 8))
plot_tree(dt_model, feature_names=feature_cols,
          class_names=['No Disease','Heart Disease'],
          filled=True, rounded=True, max_depth=3,
          fontsize=9, impurity=False)
plt.title('Decision Tree Structure (max depth 3 shown)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Decision tree visualization saved')

---
## Summary

| Component | Details |
|-----------|---------|
| **Dataset** | 303 patients across 10 Pakistan cities |
| **Model 1** | Logistic Regression — clinical + geospatial features |
| **Model 2** | Decision Tree (max_depth=5) |
| **Evaluation** | Accuracy, Confusion Matrix, ROC-AUC, Feature Importance |
| **GeoAI Map** | Interactive Folium risk map across Pakistan cities |
| **Geospatial Feature** | Urban stress index per city embedded in model |

**Key Geospatial Finding:**
- Cities with higher urban stress index (Karachi, Lahore) show higher heart disease rates
- Urban stress index ranked as an important feature in both models
- Interactive map reveals spatial clustering of risk across Pakistan

**Disclaimer:** This model is for educational purposes only. Consult a licensed cardiologist for medical diagnosis.